# 03 Resume 2D CNN v0.1

**Notebook version:** v0.1  
**Category:** training  
**Purpose:** Resume training from the most recent completed epoch using `resume_state.pt`, while launching into a new child run under the same model directory.  
**Inputs:** `./models/<model_directory>/run_register.json`, `./models/<model_directory>/runs/<run_id>/resume_state.pt`, `./src/train.py`, `./src/resume/control_panel.py`  
**Expected outputs:** new run folders under `./models/<model_directory>/runs/<run_id>/` with resumed `train.log`, `best.pt`, `latest.pt`, and `resume_state.pt`.  
**Artifact write mode:** canonical artifacts are written by `src/train.py`; this notebook orchestrates resume launch/session/log controls.  
**Decision supported:** `READY_FOR_RESUME_LAUNCH` vs `PATCH_REQUIRED`


In [ ]:
# Repo Setup
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    for parent in repo_root.parents:
        if (parent / 'src').exists() and (parent / 'models').exists():
            repo_root = parent
            break
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f'repo_root={repo_root}')


In [ ]:
# Resume Config
REPO_ROOT = repo_root
PYTHON_EXECUTABLE = sys.executable
TRAINING_MODULE = 'src.train'
MODELS_ROOT = REPO_ROOT / 'models'

DEFAULT_ADDITIONAL_EPOCHS = 4
DEFAULT_LOG_TAIL_LINES = 180
DEFAULT_LOG_POLL_SECONDS = 5.0

print(f'python={PYTHON_EXECUTABLE}')
print(f'models_root={MODELS_ROOT}')


In [ ]:
# Helper Imports
import html
import json
import threading
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

from src.resume import control_panel as resume_control

print(f'loaded_helper={resume_control.__file__}')


In [ ]:
# Resume Panel
status_area = widgets.HTML(value='')
session_suggestion = widgets.HTML(value='')

model_directory_dropdown = widgets.Dropdown(
    options=[],
    description='Model Dir',
    layout=widgets.Layout(width='460px'),
)
source_run_dropdown = widgets.Dropdown(
    options=[],
    description='Source Run',
    layout=widgets.Layout(width='460px'),
)
model_name_input = widgets.Text(
    value='fast_v0_2',
    description='Model Name',
    layout=widgets.Layout(width='320px'),
)
additional_epochs_input = widgets.BoundedIntText(
    value=DEFAULT_ADDITIONAL_EPOCHS,
    min=1,
    max=10000,
    description='Add Epochs',
    layout=widgets.Layout(width='230px'),
)
session_name_input = widgets.Text(
    value='',
    description='Session Name',
    placeholder='auto-generated from model + run id',
    layout=widgets.Layout(width='460px'),
)

source_run_dir_view = widgets.Text(
    value='',
    description='Source Dir',
    disabled=True,
    layout=widgets.Layout(width='100%'),
)
source_epoch_view = widgets.Text(
    value='',
    description='Last Epoch',
    disabled=True,
    layout=widgets.Layout(width='260px'),
)

change_note_input = widgets.Text(
    value='tmux resume launch via notebook v0.1',
    description='Change Note',
    layout=widgets.Layout(width='760px'),
)
primary_variable_input = widgets.Text(
    value='resume training from latest epoch',
    description='Primary Var',
    layout=widgets.Layout(width='760px'),
)
notes_input = widgets.Text(
    value='',
    description='Notes',
    layout=widgets.Layout(width='760px'),
)

run_id_preview = widgets.Text(
    value='',
    description='Run ID',
    disabled=True,
    layout=widgets.Layout(width='320px'),
)
run_register_path = widgets.Text(
    value='',
    description='Run Register',
    disabled=True,
    layout=widgets.Layout(width='100%'),
)
derived_log_path = widgets.Text(
    value='',
    description='Log Path',
    disabled=True,
    layout=widgets.Layout(width='100%'),
)

refresh_models_button = widgets.Button(description='Refresh Models')
refresh_sources_button = widgets.Button(description='Refresh Sources')
launch_button = widgets.Button(description='Launch Resume', button_style='success')

session_select = widgets.Dropdown(
    options=[],
    value=None,
    description='Sessions',
    layout=widgets.Layout(width='460px'),
)
refresh_sessions_button = widgets.Button(description='Refresh Sessions')
end_session_button = widgets.Button(description='End Session', button_style='danger')

log_tail_lines_input = widgets.IntText(
    value=DEFAULT_LOG_TAIL_LINES,
    description='Tail Lines',
    layout=widgets.Layout(width='220px'),
)
poll_interval_input = widgets.FloatText(
    value=DEFAULT_LOG_POLL_SECONDS,
    description='Poll Secs',
    layout=widgets.Layout(width='220px'),
)
auto_refresh_toggle = widgets.ToggleButton(
    value=False,
    description='Auto Refresh',
    icon='refresh',
    layout=widgets.Layout(width='180px'),
)
refresh_log_button = widgets.Button(description='Refresh Log', button_style='info')

log_output = widgets.Textarea(
    value='',
    description='',
    layout=widgets.Layout(width='100%', height='420px'),
    disabled=True,
)

_auto_refresh_state = {'thread': None, 'stop_event': None}
_session_name_state = {'last_auto': ''}
_source_state = {'by_run_id': {}, 'last_auto_model_name': ''}


def _set_status(message: str, *, is_error: bool = False) -> None:
    color = '#9b1c1c' if is_error else '#0f5132'
    status_area.value = (
        f"<div style='border:1px solid #d0d7de;padding:8px;border-radius:6px;'>"
        f"<strong style='color:{color};'>Status</strong>"
        f"<div>{html.escape(message)}</div>"
        '</div>'
    )


def _sync_default_session_name(suggested_session: str) -> None:
    current = session_name_input.value.strip()
    last_auto = _session_name_state['last_auto']
    if (not current) or (current == last_auto):
        session_name_input.value = suggested_session
    _session_name_state['last_auto'] = suggested_session


def _selected_source_row():
    run_id = source_run_dropdown.value
    if not run_id:
        return None
    return _source_state['by_run_id'].get(str(run_id))


def _refresh_models(*_):
    try:
        model_dirs = resume_control.list_model_directories(MODELS_ROOT)
    except Exception as exc:
        model_directory_dropdown.options = []
        model_directory_dropdown.value = None
        _set_status(f'Model directory refresh failed: {exc}', is_error=True)
        return

    current = model_directory_dropdown.value
    model_directory_dropdown.options = model_dirs
    if model_dirs:
        if current in model_dirs:
            model_directory_dropdown.value = current
        else:
            model_directory_dropdown.value = model_dirs[-1]
    else:
        model_directory_dropdown.value = None


def _update_preview_paths(show_error: bool = False):
    model_directory = model_directory_dropdown.value
    if not model_directory:
        run_id_preview.value = ''
        run_register_path.value = ''
        derived_log_path.value = ''
        session_suggestion.value = '<em>Suggested session name: <code>rb-resume-&lt;run_id&gt;</code></em>'
        return None

    try:
        next_run_id = resume_control.preview_next_run_id(
            models_root=MODELS_ROOT,
            model_directory=model_directory,
        )
        run_id_preview.value = next_run_id

        model_dir = resume_control.build_model_dir(
            models_root=MODELS_ROOT,
            model_directory=model_directory,
        )
        register_path = model_dir / 'run_register.json'
        run_register_path.value = str(register_path)

        log_path = model_dir / 'runs' / next_run_id / 'train.log'
        derived_log_path.value = str(log_path)

        suggested_session = f'rb-resume-{next_run_id}'
        session_suggestion.value = f"<em>Suggested session name: <code>{html.escape(suggested_session)}</code></em>"
        _sync_default_session_name(suggested_session)

        return {
            'model_directory': model_directory,
            'run_id': next_run_id,
            'run_register_path': str(register_path),
            'log_path': str(log_path),
            'suggested_session': suggested_session,
        }
    except Exception as exc:
        run_id_preview.value = ''
        run_register_path.value = ''
        derived_log_path.value = ''
        session_suggestion.value = '<em>Suggested session name: <code>rb-resume-&lt;run_id&gt;</code></em>'
        if show_error:
            _set_status(f'Path derivation failed: {exc}', is_error=True)
        return None


def _refresh_source_runs(*_):
    model_directory = model_directory_dropdown.value
    if not model_directory:
        source_run_dropdown.options = []
        source_run_dropdown.value = None
        _source_state['by_run_id'] = {}
        source_run_dir_view.value = ''
        source_epoch_view.value = ''
        return

    try:
        rows = resume_control.list_resume_candidates(
            models_root=MODELS_ROOT,
            model_directory=model_directory,
        )
    except Exception as exc:
        source_run_dropdown.options = []
        source_run_dropdown.value = None
        _source_state['by_run_id'] = {}
        _set_status(f'Source run refresh failed: {exc}', is_error=True)
        return

    resumable = [row for row in rows if bool(row.get('is_resumable'))]
    _source_state['by_run_id'] = {str(row['run_id']): row for row in resumable}

    options = [
        (
            f"{row['run_id']} | epoch {row['last_completed_epoch']} | {row.get('updated_at') or 'time unknown'}",
            str(row['run_id']),
        )
        for row in resumable
    ]

    current = source_run_dropdown.value
    source_run_dropdown.options = options
    if options:
        values = [value for _, value in options]
        source_run_dropdown.value = current if current in values else values[-1]
    else:
        source_run_dropdown.value = None

    _refresh_source_details()


def _refresh_source_details(*_):
    row = _selected_source_row()
    if row is None:
        source_run_dir_view.value = ''
        source_epoch_view.value = ''
        return

    source_run_dir_view.value = str(row['run_dir'])
    source_epoch_view.value = str(row.get('last_completed_epoch') or '')

    try:
        config_path = Path(row['run_dir']) / 'config.json'
        with config_path.open('r', encoding='utf-8') as handle:
            source_config = json.load(handle)
        variant = str(source_config.get('model_architecture_variant', '')).strip()
        if variant:
            model_name_input.value = variant
            _source_state['last_auto_model_name'] = variant
    except Exception:
        pass


def _refresh_sessions(*, preferred: str | None = None) -> None:
    try:
        sessions = resume_control.list_sessions()
    except Exception as exc:
        session_select.options = []
        session_select.value = None
        _set_status(f'Session refresh failed: {exc}', is_error=True)
        return

    current = preferred if preferred is not None else session_select.value
    session_select.options = sessions
    if sessions:
        if current in sessions:
            session_select.value = current
        else:
            session_select.value = sessions[0]
    else:
        session_select.value = None


def _resolve_log_path_from_selected_session() -> str | None:
    selected = session_select.value or session_name_input.value.strip()
    if not selected:
        preview = _update_preview_paths(show_error=False)
        return None if preview is None else preview['log_path']

    try:
        resolved = resume_control.resolve_session_run(
            session_name=selected,
            models_root=MODELS_ROOT,
        )
    except Exception as exc:
        _set_status(f'Session lookup failed: {exc}', is_error=True)
        return None

    if resolved is None:
        return None

    model_directory_dropdown.value = resolved['model_directory']
    run_id_preview.value = resolved['run_id']
    derived_log_path.value = resolved['log_path']
    return str(resolved['log_path'])


def _refresh_log(*_):
    selected_session = session_select.value or session_name_input.value.strip()
    log_path = _resolve_log_path_from_selected_session()
    if log_path is None:
        if selected_session:
            log_output.value = f'[no registered run for session: {selected_session}]'
        else:
            log_output.value = '[no session selected]'
        return

    try:
        tail_lines = int(log_tail_lines_input.value)
        log_output.value = resume_control.read_log_tail(log_path, max_lines_or_chars=tail_lines)
    except Exception as exc:
        log_output.value = ''
        _set_status(f'Log refresh failed: {exc}', is_error=True)


def _stop_auto_refresh() -> None:
    stop_event = _auto_refresh_state.get('stop_event')
    thread = _auto_refresh_state.get('thread')
    if stop_event is not None:
        stop_event.set()
    if thread is not None and thread.is_alive():
        thread.join(timeout=1.0)
    _auto_refresh_state['thread'] = None
    _auto_refresh_state['stop_event'] = None


def _auto_refresh_loop(stop_event: threading.Event) -> None:
    while not stop_event.is_set():
        try:
            _refresh_log()
        except Exception as exc:
            _set_status(f'Auto refresh failed: {exc}', is_error=True)
        interval = max(0.5, float(poll_interval_input.value))
        stop_event.wait(interval)


def _on_auto_refresh_toggle(change):
    enabled = bool(change['new'])
    if enabled:
        _stop_auto_refresh()
        stop_event = threading.Event()
        thread = threading.Thread(target=_auto_refresh_loop, args=(stop_event,), daemon=True)
        _auto_refresh_state['thread'] = thread
        _auto_refresh_state['stop_event'] = stop_event
        thread.start()
        _set_status('Auto refresh enabled.')
    else:
        _stop_auto_refresh()
        _set_status('Auto refresh disabled.')


def _on_launch_clicked(_):
    preview = _update_preview_paths(show_error=True)
    if preview is None:
        return

    source_row = _selected_source_row()
    if source_row is None:
        _set_status('No resumable source run selected.', is_error=True)
        return

    session_name = session_name_input.value.strip()
    if not session_name:
        _set_status(
            f"Session name is required. Suggested: {preview['suggested_session']}",
            is_error=True,
        )
        return

    if resume_control.session_exists(session_name):
        _set_status(f'Duplicate session name refused: {session_name}', is_error=True)
        return

    try:
        reservation = resume_control.reserve_resume_run(
            models_root=MODELS_ROOT,
            model_directory=preview['model_directory'],
            model_name=model_name_input.value.strip() or 'resume',
            session_name=session_name,
            source_run_id=source_row['run_id'],
            primary_variable_changed=primary_variable_input.value.strip(),
            notes=notes_input.value.strip(),
        )

        run_id_preview.value = reservation['run_id']
        run_register_path.value = reservation['run_register_path']
        derived_log_path.value = reservation['log_path']

        command = resume_control.build_resume_launch_command(
            run_id=reservation['run_id'],
            model_directory=preview['model_directory'],
            source_run_dir=source_row['run_dir'],
            additional_epochs=int(additional_epochs_input.value),
            python_executable=PYTHON_EXECUTABLE,
            training_module=TRAINING_MODULE,
            output_root=MODELS_ROOT,
            change_note=change_note_input.value.strip() or 'resume launch',
        )

        resume_control.launch_session(
            session_name=session_name,
            command=command,
            log_path=reservation['log_path'],
            working_directory=REPO_ROOT,
        )

        _refresh_sessions(preferred=session_name)
        _refresh_log()
        _set_status(
            f"Launched resume session {session_name} from {source_row['run_id']} "
            f"-> {reservation['run_id']} (+{int(additional_epochs_input.value)} epochs)"
        )
    except Exception as exc:
        _set_status(f'Resume launch failed: {exc}', is_error=True)


def _on_end_session_clicked(_):
    selected_session = session_select.value
    if not selected_session:
        _set_status('Select a session to end.', is_error=True)
        return
    try:
        ended = resume_control.end_session(selected_session)
        if ended:
            _set_status(f'Ended session {selected_session}.')
        else:
            _set_status(f'Session not found: {selected_session}', is_error=True)
        _refresh_sessions()
        _refresh_log()
    except Exception as exc:
        _set_status(f'End session failed: {exc}', is_error=True)


def _on_session_changed(change):
    selected = change['new']
    if selected:
        session_name_input.value = selected
    _refresh_log()


def _on_model_directory_changed(_):
    _update_preview_paths(show_error=False)
    _refresh_source_runs()


launch_button.on_click(_on_launch_clicked)
end_session_button.on_click(_on_end_session_clicked)
refresh_models_button.on_click(_refresh_models)
refresh_sources_button.on_click(_refresh_source_runs)
refresh_sessions_button.on_click(lambda _: _refresh_sessions())
refresh_log_button.on_click(_refresh_log)
session_select.observe(_on_session_changed, names='value')
model_directory_dropdown.observe(_on_model_directory_changed, names='value')
source_run_dropdown.observe(_refresh_source_details, names='value')
auto_refresh_toggle.observe(_on_auto_refresh_toggle, names='value')

_refresh_models()
_update_preview_paths(show_error=False)
_refresh_source_runs()
_refresh_sessions()
_set_status('Resume panel ready.')

display(widgets.VBox([
    status_area,
    widgets.HBox([model_directory_dropdown, refresh_models_button, refresh_sources_button]),
    widgets.HBox([source_run_dropdown, source_epoch_view]),
    source_run_dir_view,
    widgets.HBox([model_name_input, additional_epochs_input]),
    session_name_input,
    session_suggestion,
    change_note_input,
    primary_variable_input,
    notes_input,
    widgets.HBox([run_id_preview]),
    run_register_path,
    derived_log_path,
    widgets.HBox([launch_button, refresh_sessions_button, end_session_button]),
    session_select,
    widgets.HBox([refresh_log_button, log_tail_lines_input, poll_interval_input, auto_refresh_toggle]),
    log_output,
]))


## Final Verdict
`READY_FOR_RESUME_LAUNCH` when the panel resolves a resumable source run (`resume_state.pt` present), reserves a child run, launches tmux, and tails the new run log.
